<div align="center">
<h1 style="color: PURPLE; font-weight:700;text-decoration:underline;"> ~ CAPSTONE PROJECT 3~ </h1>
</div>

<h1 style="color: #2E86C1; text-align: center;">Emotion Detection Project</h1>

This notebook implements a complete pipeline for Facial Expression Recognition (FER) using a Deep Learning and Computer Vision approach with TensorFlow and Keras.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

<h2 style="color: #E67E22;">1. Data Preparation</h2>
Set the paths for the training and testing datasets.

In [ ]:
train_dir = 'emotion-detection-fer/train/'
test_dir = 'emotion-detection-fer/test/'

classes = os.listdir(train_dir)
print(f"Classes: {classes}")

<h2 style="color: #8E44AD;">2. Exploratory Data Analysis (EDA)</h2>
<h3 style="color: #3498DB;">Visualize Sample Images</h3>

In [ ]:
plt.figure(figsize=(15, 10))
for i, emotion in enumerate(classes):
    img_list = os.listdir(os.path.join(train_dir, emotion))
    if img_list:
        img_path = os.path.join(train_dir, emotion, img_list[0])
        img = tf.keras.preprocessing.image.load_img(img_path, target_size=(48, 48))
        plt.subplot(1, 7, i+1)
        plt.imshow(img)
        plt.title(emotion)
        plt.axis('off')
plt.show()

<h3 style="color: #3498DB;">Class Distribution</h3>

In [ ]:
train_counts = {emotion: len(os.listdir(os.path.join(train_dir, emotion))) for emotion in classes}
test_counts = {emotion: len(os.listdir(os.path.join(test_dir, emotion))) for emotion in classes}

df_counts = pd.DataFrame({'Train': train_counts, 'Test': test_counts})
df_counts.plot(kind='bar', figsize=(10, 6))
plt.title('Class Distribution')
plt.ylabel('Number of Images')
plt.show()

<h2 style="color: #16A085;">3. Data Preprocessing</h2>

We use `ImageDataGenerator` for data augmentation and normalization. 
**Note:** To speed up the run, we are using a 10% subset of the data.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.9
)

test_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.9)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(48, 48),
    batch_size=64,
    class_mode='categorical',
    color_mode='grayscale',
    subset='training'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(48, 48),
    batch_size=64,
    class_mode='categorical',
    color_mode='grayscale',
    shuffle=False,
    subset='training'
)

<h2 style="color: #C0392B;">4. Model Building</h2>
We build a Sequential CNN model.

In [ ]:
model = Sequential([
    Conv2D(64, (3, 3), activation='relu', input_shape=(48, 48, 1)),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    Flatten(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(7, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

<h2 style="color: #D35400;">5. Training</h2>
Define callbacks and train the model.

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    callbacks=[early_stopping, reduce_lr]
)

<h2 style="color: #27AE60;">6. Evaluation</h2>
Plot training history and generate performance reports.

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.show()

In [ ]:
y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes

print("Classification Report:")
print(classification_report(y_true, y_pred_classes, target_names=classes))

cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
model.save('emotion_model.h5')
print("Model saved as emotion_model.h5")

<h2 style="color: #2980B9;">7. Real-time Inference & Face Detection</h2>
In this section, we use OpenCV's Haar Cascade to detect faces and then use our trained model to predict emotions.

In [11]:
# Load Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def predict_emotion(img_path):
    # Load the image
    img = cv2.imread(img_path)
    if img is None:
        print("Image not found. Please check the path.")
        return
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Detect faces
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    for (x, y, w, h) in faces:
        # Extract face and preprocess
        roi_gray = gray[y:y+h, x:x+w]
        roi_gray = cv2.resize(roi_gray, (48, 48))
        roi_gray = roi_gray.astype('float32') / 255.0
        roi_gray = np.expand_dims(roi_gray, axis=0)
        roi_gray = np.expand_dims(roi_gray, axis=-1)
        
        # Predict emotion
        prediction = model.predict(roi_gray)
        maxindex = int(np.argmax(prediction))
        predicted_emotion = classes[maxindex]
        
        # Draw bounding box and label
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(img, predicted_emotion, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36,255,12), 2)
    
    # Convert BGR to RGB for plotting
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.show()

<h2 style="color: #7D3C98;">8. Prediction on Custom Image</h2>
Enter your image path in the cell below and run it. Other method is to keep the image in the same folder where the project is being run. In that case, instead of image path , image name should be given. 

In [ ]:
# Example: replace 'my_photo.jpg' with your filename
predict_emotion(r'C:\Users\Tatai & Papai\OneDrive\Desktop\my_photo.jpg')

-----------